In [2]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report, roc_curve, auc
from peft import LoraConfig, get_peft_model  # For efficient fine-tuning



In [3]:
df = pd.read_excel('bioactivity_dataset_cleaned_outliers.xlsx')
df['mol'] = df['canonical_smiles'].apply(Chem.MolFromSmiles)
df = df.dropna(subset=['mol'])

# Compute/scale descriptors (if not already)
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return np.zeros(4)
    return np.array([Descriptors.MolWt(mol),
                Descriptors.MolLogP(mol),
                Descriptors.TPSA(mol),
                Descriptors.NumHDonors(mol),
                Descriptors.NumHAcceptors(mol),
                Descriptors.NumRotatableBonds(mol),
                Descriptors.NumAromaticRings(mol),
                Descriptors.FractionCSP3(mol)])

df[['MW', 'LogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'NumRotatableBonds', 'NumAromaticRings', 'FractionCSP3']] = df['canonical_smiles'].apply(compute_descriptors).tolist()

scaler = StandardScaler()
desc_cols = ['MW', 'LogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'NumRotatableBonds', 'NumAromaticRings', 'FractionCSP3']
df[desc_cols] = scaler.fit_transform(df[desc_cols])

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5667 entries, 0 to 5666
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   canonical_smiles   5667 non-null   object 
 1   MW                 5667 non-null   float64
 2   LogP               5667 non-null   float64
 3   NumHDonors         5667 non-null   float64
 4   NumHAcceptors      5667 non-null   float64
 5   pIC50              5667 non-null   float64
 6   bioactivity_class  5667 non-null   object 
 7   bioactivity        5667 non-null   int64  
 8   mol                5667 non-null   object 
 9   TPSA               5667 non-null   float64
 10  NumRotatableBonds  5667 non-null   float64
 11  NumAromaticRings   5667 non-null   float64
 12  FractionCSP3       5667 non-null   float64
dtypes: float64(9), int64(1), object(3)
memory usage: 575.7+ KB


In [5]:
df.head(3)

,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors,pIC50,bioactivity_class,bioactivity,mol,TPSA,NumRotatableBonds,NumAromaticRings,FractionCSP3
0,O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc...,0.524109,0.241522,1.32487,0.282913,8.6,active,1,<rdkit.Chem.rdchem.Mol object at 0x00000282036...,1.117016,1.146765,0.274162,-0.064181
1,O=C(CCCCCC(C(=O)Nc1ccc2ncccc2c1)C(=O)Nc1ccc2nc...,0.732961,0.621072,1.32487,0.282913,9.0,active,1,<rdkit.Chem.rdchem.Mol object at 0x00000282036...,1.243444,0.815502,1.181684,-0.403666
2,O=C(/C=C/c1cccc(C(C(=O)Nc2ccccc2)C(=O)Nc2ccccc...,0.036571,-0.025903,1.32487,-0.713216,9.0,active,1,<rdkit.Chem.rdchem.Mol object at 0x00000282036...,0.352922,-0.178287,0.274162,-1.464556


In [6]:
df.drop(columns=['pIC50', 'bioactivity_class'], inplace=True)

In [7]:
df.iloc[0]['canonical_smiles']

'O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc12)NO'

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5667 entries, 0 to 5666
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   canonical_smiles   5667 non-null   object 
 1   MW                 5667 non-null   float64
 2   LogP               5667 non-null   float64
 3   NumHDonors         5667 non-null   float64
 4   NumHAcceptors      5667 non-null   float64
 5   bioactivity        5667 non-null   int64  
 6   mol                5667 non-null   object 
 7   TPSA               5667 non-null   float64
 8   NumRotatableBonds  5667 non-null   float64
 9   NumAromaticRings   5667 non-null   float64
 10  FractionCSP3       5667 non-null   float64
dtypes: float64(8), int64(1), object(2)
memory usage: 487.1+ KB


In [9]:
df['bioactivity'].value_counts()

bioactivity
1    4641
0    1026
Name: count, dtype: int64

In [10]:
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTENC


train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['bioactivity'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

tokenizer = AutoTokenizer.from_pretrained('ibm/MoLFormer-XL-both-10pct', trust_remote_code=True)

class BioactivityDataset(Dataset):
    def __init__(self, df, tokenizer, desc_scaler):
        self.smiles = df['canonical_smiles'].tolist()
        self.labels = df['bioactivity'].tolist()
        self.descs = df[desc_cols].values
        #self.desc_scaler = desc_scaler
        self.encodings = tokenizer(self.smiles, truncation=True, padding=True, max_length=512, return_tensors='pt')
    
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        item['descriptors'] = torch.tensor(self.descs[idx], dtype=torch.float)
        return item

In [11]:
train_dataset = BioactivityDataset(train_df, tokenizer, scaler)
val_dataset = BioactivityDataset(val_df, tokenizer, scaler)
test_dataset = BioactivityDataset(test_df, tokenizer, scaler)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [12]:
train_df['bioactivity'].value_counts(), val_df['bioactivity'].value_counts(), test_df['bioactivity'].value_counts()

(bioactivity
 1    3712
 0     821
 Name: count, dtype: int64,
 bioactivity
 1    454
 0    113
 Name: count, dtype: int64,
 bioactivity
 1    475
 0     92
 Name: count, dtype: int64)

In [13]:
train_dataset[0]

{'input_ids': tensor([ 0,  4, 10,  4,  6, 12,  9,  7, 26,  4,  6,  4,  4,  4,  4,  4,  4,  4,
          6, 12,  9,  7, 10,  5,  8,  5,  5,  5,  5,  6, 21,  5, 11,  5,  5,  5,
          5,  5, 11,  7,  5,  8,  7, 12, 10, 33,  9,  1,  2,  2,  2,  2,  2,  2,
          2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
          2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
          2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
          2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
          2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
          2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
          2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
          2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       

In [14]:
class FineTunedMultimodalModel(nn.Module):
    def __init__(self, num_desc=8, emb_dim=768, hidden_dim=512, dropout=0.5, num_layers_to_unfreeze=4):
        super().__init__()
        self.smiles_encoder = AutoModel.from_pretrained('ibm/MoLFormer-XL-both-10pct', trust_remote_code=True)

        self.config = self.smiles_encoder.config
        self.config.num_labels = 1  # Optional: Set for classification task

        # Freeze all except last num_layers_to_unfreeze
        for param in self.smiles_encoder.parameters():
            param.requires_grad = False
        unfreeze_modules = self.smiles_encoder.encoder.layer[-num_layers_to_unfreeze:]
        for module in unfreeze_modules:
            for param in module.parameters():
                param.requires_grad = True
        
        self.desc_proj = nn.Linear(num_desc, emb_dim)
        self.layer_norm = nn.LayerNorm(emb_dim * 2)  # New: Normalize fused embeddings

        self.fusion = nn.Sequential(
            nn.Linear(emb_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),  # New: Helps with internal covariate shift
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),  # New: Extra layer for complexity
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1)  # Logits output (no sigmoid)
        )

    def forward(self, input_ids=None, attention_mask=None, descriptors=None, **kwargs):
        outputs = self.smiles_encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        
        # Improved pooling: Mean over sequence (better than CLS for molecules)
        smiles_emb = torch.mean(outputs.last_hidden_state, dim=1)  # [batch, emb_dim]
        
        desc_emb = self.desc_proj(descriptors)
        fused = torch.cat([smiles_emb, desc_emb], dim=-1)
        fused = self.layer_norm(fused)  # New: Stabilizes fusion
        
        return self.fusion(fused).squeeze(-1)  # Return logits
    
model = FineTunedMultimodalModel(num_layers_to_unfreeze=2)  # Start with 4 for small data
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

FineTunedMultimodalModel(
  (smiles_encoder): MolformerModel(
    (embeddings): MolformerEmbeddings(
      (word_embeddings): Embedding(2362, 768, padding_idx=2)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (encoder): MolformerEncoder(
      (layer): ModuleList(
        (0-11): 12 x MolformerLayer(
          (attention): MolformerAttention(
            (self): MolformerSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (rotary_embeddings): MolformerRotaryEmbedding()
              (feature_map): MolformerFeatureMap(
                (kernel): ReLU()
              )
            )
            (output): MolformerSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)


In [75]:
class AttentionFusion(nn.Module):
    """
    Cross-attention between SMILES and descriptors
    This learns which parts of each modality are most relevant
    """
    def __init__(self, emb_dim=768, num_heads=8, dropout=0.3):
        super().__init__()
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=emb_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.layer_norm = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, smiles_emb, desc_emb):
        # Cross-attend: descriptors attend to SMILES
        desc_emb = desc_emb.unsqueeze(1)  # [batch, 1, emb_dim]
        smiles_emb = smiles_emb.unsqueeze(1)  # [batch, 1, emb_dim]
        
        attended, _ = self.cross_attention(
            query=desc_emb,
            key=smiles_emb,
            value=smiles_emb
        )
        
        # Residual connection
        fused = self.layer_norm(desc_emb + self.dropout(attended))
        return fused.squeeze(1)


class ImprovedMultimodalModel(nn.Module):
    """
    Improvement Strategy 1: Better fusion with attention
    Expected gain: +1-2% accuracy
    """
    def __init__(self, num_desc=8, emb_dim=768, hidden_dim=512, 
                 dropout=0.3, num_layers_to_unfreeze=3):
        super().__init__()
        
        # ChemBERTa encoder
        self.smiles_encoder = AutoModel.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
        
        # Freeze all except last layers
        for param in self.smiles_encoder.parameters():
            param.requires_grad = False
        
        # Unfreeze last N transformer layers
        unfreeze_modules = self.smiles_encoder.encoder.layer[-num_layers_to_unfreeze:]
        for module in unfreeze_modules:
            for param in module.parameters():
                param.requires_grad = True
        
        # Enhanced descriptor processing
        self.desc_proj = nn.Sequential(
            nn.Linear(num_desc, emb_dim // 2),
            nn.LayerNorm(emb_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(emb_dim // 2, emb_dim),
            nn.LayerNorm(emb_dim)
        )
        
        # Attention-based fusion
        self.attention_fusion = AttentionFusion(emb_dim, num_heads=8, dropout=dropout)
        
        # Classification head with residual connections
        self.classifier = nn.Sequential(
            nn.Linear(emb_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(dropout / 2),  # Less dropout in final layers
            nn.Linear(hidden_dim // 4, 1)
        )
        
    def forward(self, input_ids=None, attention_mask=None, descriptors=None, **kwargs):
        # SMILES encoding
        outputs = self.smiles_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            **kwargs
        )
        
        # Use [CLS] token
        smiles_emb = outputs.last_hidden_state[:, 0, :]
        
        # Descriptor encoding
        desc_emb = self.desc_proj(descriptors)
        
        # Attention fusion
        fused_desc = self.attention_fusion(smiles_emb, desc_emb)
        
        # Concatenate for final prediction
        combined = torch.cat([smiles_emb, fused_desc], dim=-1)
        
        return self.classifier(combined).squeeze(-1)

model = ImprovedMultimodalModel(num_desc=4, num_layers_to_unfreeze=2)
model.to(device)

ImprovedMultimodalModel(
  (smiles_encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(767, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (

In [34]:
class ChemBERTaClassifier(nn.Module):
    def __init__(self, model_name, num_num_feats=4, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.fc_concat = nn.Linear(self.bert.config.hidden_size + num_num_feats, 128)
        self.fc_out = nn.Linear(128, 1)

    def forward(self, input_ids, attention_mask, num_feats):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output  # [CLS] token representation
        concat = torch.cat([pooled, num_feats], dim=1)
        x = self.dropout(torch.relu(self.fc_concat(concat)))
        return torch.sigmoid(self.fc_out(x)).squeeze()

In [39]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig( task_type=TaskType.SEQ_CLS, r=4, lora_alpha=8, target_modules=["query", "value"], lora_dropout=0.1)

In [40]:
model = get_peft_model(model, lora_config)
 
# Print trainable params (should be <1% of total)
model.print_trainable_parameters()  

trainable params: 1,026,561 || all params: 48,747,138 || trainable%: 2.1059


In [15]:
from sklearn.utils.class_weight import compute_class_weight
import torch.optim as optim

class_weights = compute_class_weight('balanced', classes=np.unique(train_df['bioactivity']), y=train_df['bioactivity'])
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

pos_weight = torch.tensor([3712 / 821]).to(device)  # From your imbalance
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)  # Added weight decay
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)  # New: Better convergence

# criterion = BCEWithLogitsLoss(pos_weight=class_weights[1])
epochs = 50  # Longer for fine-tuning
best_auc = 0
patience, counter = 5, 0

In [16]:
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        descriptors = batch['descriptors'].to(device)
        labels = batch['labels'].float().to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask, descriptors=descriptors)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    
    # Validation
    model.eval()
    val_preds, val_labels = [], []
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            descriptors = batch['descriptors'].to(device)
            labels = batch['labels'].to(device)
            logits = model(input_ids, attention_mask, descriptors)
            probs = torch.sigmoid(logits).cpu().numpy()
            val_preds.extend(probs)
            val_labels.extend(labels.cpu().numpy())
            loss = criterion(logits, labels.float())  # Compute loss for this batch
            val_loss += loss.item()  #
    
    scheduler.step(val_loss / len(val_loader))  # Adjust LR
    
    auc = roc_auc_score(val_labels, val_preds)
    print(f'Epoch {epoch+1}: Train Loss {train_loss/len(train_loader):.4f}, Val AUROC {auc:.4f}')
    
    if auc > best_auc:
        best_auc = auc
        torch.save(model, 'best_finetuned_model_.pth')
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping!")
            break

Epoch 1: Train Loss 2.1565, Val AUROC 0.7490
Epoch 2: Train Loss 1.5015, Val AUROC 0.7731
Epoch 3: Train Loss 1.1263, Val AUROC 0.7929
Epoch 4: Train Loss 0.8921, Val AUROC 0.8077
Epoch 5: Train Loss 0.7559, Val AUROC 0.8301
Epoch 6: Train Loss 0.6732, Val AUROC 0.8419
Epoch 7: Train Loss 0.6179, Val AUROC 0.8553
Epoch 8: Train Loss 0.5827, Val AUROC 0.8609
Epoch 9: Train Loss 0.5494, Val AUROC 0.8649
Epoch 10: Train Loss 0.5123, Val AUROC 0.8650
Epoch 11: Train Loss 0.5021, Val AUROC 0.8797
Epoch 12: Train Loss 0.4690, Val AUROC 0.8783
Epoch 13: Train Loss 0.4537, Val AUROC 0.9022
Epoch 14: Train Loss 0.4199, Val AUROC 0.8848
Epoch 15: Train Loss 0.4060, Val AUROC 0.9136
Epoch 16: Train Loss 0.3902, Val AUROC 0.9067
Epoch 17: Train Loss 0.3737, Val AUROC 0.9202
Epoch 18: Train Loss 0.3537, Val AUROC 0.9293
Epoch 19: Train Loss 0.3401, Val AUROC 0.9179
Epoch 20: Train Loss 0.3281, Val AUROC 0.9284
Epoch 21: Train Loss 0.3090, Val AUROC 0.9288
Epoch 22: Train Loss 0.3039, Val AUROC 0.92

In [22]:
loaded_model = torch.load('best_finetuned_model_.pth', map_location=device, weights_only=False)
loaded_model.eval()


FineTunedMultimodalModel(
  (smiles_encoder): MolformerModel(
    (embeddings): MolformerEmbeddings(
      (word_embeddings): Embedding(2362, 768, padding_idx=2)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (encoder): MolformerEncoder(
      (layer): ModuleList(
        (0-11): 12 x MolformerLayer(
          (attention): MolformerAttention(
            (self): MolformerSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (rotary_embeddings): MolformerRotaryEmbedding()
              (feature_map): MolformerFeatureMap(
                (kernel): ReLU()
              )
            )
            (output): MolformerSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)


In [17]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Run inference on test set
model.eval()
test_preds = []
test_labels = []
test_probs = []
with torch.no_grad():
	for batch in test_loader:
		input_ids = batch['input_ids'].to(device)
		attention_mask = batch['attention_mask'].to(device)
		descriptors = batch['descriptors'].to(device)
		labels = batch['labels'].to(device)
		logits = model(input_ids, attention_mask, descriptors)
		probs = torch.sigmoid(logits).cpu().numpy()
		test_probs.extend(probs)
		test_preds.extend((probs > 0.5).astype(int))
		test_labels.extend(labels.cpu().numpy())

accuracy = balanced_accuracy_score(test_labels, test_preds)
auc = roc_auc_score(test_labels, test_probs)
print(f"AUC: {auc}")
print(f"Accuracy: {accuracy}")
print("accuracy another: ", accuracy_score(test_labels, test_preds))

AUC: 0.9513272311212815
Accuracy: 0.8611441647597253
accuracy another:  0.9435626102292769


In [105]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

test_preds = []
test_labels = []
test_probs = []
with torch.no_grad():
	for batch in test_loader:
		input_ids = batch['input_ids'].to(device)
		attention_mask = batch['attention_mask'].to(device)
		descriptors = batch['descriptors'].to(device)
		labels = batch['labels'].to(device)
		logits = loaded_model(input_ids, attention_mask, descriptors)
		probs = torch.sigmoid(logits).cpu().numpy()
		test_probs.extend(probs)
		test_preds.extend((probs > 0.5).astype(int))
		test_labels.extend(labels.cpu().numpy())

accuracy = balanced_accuracy_score(test_labels, test_preds)
auc = roc_auc_score(test_labels, test_probs)
print(f"AUC: {auc}")
print(f"Accuracy: {accuracy}")
print("accuracy another: ", accuracy_score(test_labels, test_preds))

AUC: 0.9454919908466819
Accuracy: 0.8548283752860412
accuracy another:  0.9329805996472663


In [18]:
print(classification_report(test_labels, test_preds))

              precision    recall  f1-score   support

           0       0.89      0.74      0.81        92
           1       0.95      0.98      0.97       475

    accuracy                           0.94       567
   macro avg       0.92      0.86      0.89       567
weighted avg       0.94      0.94      0.94       567



In [19]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, confusion_matrix, average_precision_score
import numpy as np

y_true = np.array(test_labels)
y_pred = np.array(test_preds)
y_prob = np.array(test_probs)

accuracy = accuracy_score(y_true, y_pred)
balanced_acc = balanced_accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_prob)
mcc = matthews_corrcoef(y_true, y_pred)
auprc = average_precision_score(y_true, y_prob)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
specificity = tn / (tn + fp)
sensitivity = recall  # same as recall

print(f"Accuracy: {accuracy:.4f}")
print(f"Balanced Acc.: {balanced_acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"MCC: {mcc:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"AUPRC: {auprc:.4f}")

Accuracy: 0.9436
Balanced Acc.: 0.8611
Precision: 0.9511
Recall: 0.9832
F1 Score: 0.9669
ROC-AUC: 0.9513
MCC: 0.7816
Specificity: 0.7391
Sensitivity: 0.9832
AUPRC: 0.9879


In [25]:
"""
Improved Multimodal Model for Bioactivity Prediction
Target: 95-97% accuracy (from current 92%)

Key Improvements:
1. Advanced fusion mechanisms (attention-based)
2. Better descriptor processing
3. Regularization techniques
4. Data augmentation for molecules
5. Ensemble predictions
6. Optimal training strategies
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
import math

# ============================================================================
# IMPROVEMENT 1: Advanced Attention-Based Fusion
# ============================================================================

class AttentionFusion(nn.Module):
    """
    Cross-attention between SMILES and descriptors
    This learns which parts of each modality are most relevant
    """
    def __init__(self, emb_dim=768, num_heads=8, dropout=0.3):
        super().__init__()
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=emb_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.layer_norm = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, smiles_emb, desc_emb):
        # Cross-attend: descriptors attend to SMILES
        desc_emb = desc_emb.unsqueeze(1)  # [batch, 1, emb_dim]
        smiles_emb = smiles_emb.unsqueeze(1)  # [batch, 1, emb_dim]
        
        attended, _ = self.cross_attention(
            query=desc_emb,
            key=smiles_emb,
            value=smiles_emb
        )
        
        # Residual connection
        fused = self.layer_norm(desc_emb + self.dropout(attended))
        return fused.squeeze(1)


class ImprovedMultimodalModel(nn.Module):
    """
    Improvement Strategy 1: Better fusion with attention
    Expected gain: +1-2% accuracy
    """
    def __init__(self, num_desc=8, emb_dim=768, hidden_dim=512, 
                 dropout=0.3, num_layers_to_unfreeze=3):
        super().__init__()
        
        # ChemBERTa encoder
        self.smiles_encoder = AutoModel.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
        
        # Freeze all except last layers
        for param in self.smiles_encoder.parameters():
            param.requires_grad = False
        
        # Unfreeze last N transformer layers
        unfreeze_modules = self.smiles_encoder.encoder.layer[-num_layers_to_unfreeze:]
        for module in unfreeze_modules:
            for param in module.parameters():
                param.requires_grad = True
        
        # Enhanced descriptor processing
        self.desc_proj = nn.Sequential(
            nn.Linear(num_desc, emb_dim // 2),
            nn.LayerNorm(emb_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(emb_dim // 2, emb_dim),
            nn.LayerNorm(emb_dim)
        )
        
        # Attention-based fusion
        self.attention_fusion = AttentionFusion(emb_dim, num_heads=8, dropout=dropout)
        
        # Classification head with residual connections
        self.classifier = nn.Sequential(
            nn.Linear(emb_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(dropout / 2),  # Less dropout in final layers
            nn.Linear(hidden_dim // 4, 1)
        )
        
    def forward(self, input_ids=None, attention_mask=None, descriptors=None, **kwargs):
        # SMILES encoding
        outputs = self.smiles_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            **kwargs
        )
        
        # Use [CLS] token
        smiles_emb = outputs.last_hidden_state[:, 0, :]
        
        # Descriptor encoding
        desc_emb = self.desc_proj(descriptors)
        
        # Attention fusion
        fused_desc = self.attention_fusion(smiles_emb, desc_emb)
        
        # Concatenate for final prediction
        combined = torch.cat([smiles_emb, fused_desc], dim=-1)
        
        return self.classifier(combined).squeeze(-1)


# ============================================================================
# IMPROVEMENT 2: Multi-Scale Feature Extraction
# ============================================================================

class MultiScaleMultimodalModel(nn.Module):
    """
    Improvement Strategy 2: Use multiple ChemBERTa layers
    Expected gain: +1-1.5% accuracy
    """
    def __init__(self, num_desc=8, emb_dim=768, hidden_dim=512, 
                 dropout=0.3, num_layers_to_unfreeze=4):
        super().__init__()
        
        self.smiles_encoder = AutoModel.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
        
        # Freeze and unfreeze
        for param in self.smiles_encoder.parameters():
            param.requires_grad = False
        unfreeze_modules = self.smiles_encoder.encoder.layer[-num_layers_to_unfreeze:]
        for module in unfreeze_modules:
            for param in module.parameters():
                param.requires_grad = True
        
        # Descriptor processing
        self.desc_proj = nn.Sequential(
            nn.Linear(num_desc, emb_dim // 2),
            nn.LayerNorm(emb_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(emb_dim // 2, emb_dim),
            nn.LayerNorm(emb_dim)
        )
        
        # Multi-scale aggregation: use last 4 hidden states
        self.layer_weights = nn.Parameter(torch.ones(4) / 4)
        
        # Gate mechanism to control SMILES vs Descriptor importance
        self.gate = nn.Sequential(
            nn.Linear(emb_dim * 2, emb_dim),
            nn.Sigmoid()
        )
        
        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(emb_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        
    def forward(self, input_ids=None, attention_mask=None, descriptors=None, **kwargs):
        # Get all hidden states
        outputs = self.smiles_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            **kwargs
        )
        
        # Weighted combination of last 4 layers
        hidden_states = outputs.hidden_states[-4:]  # Last 4 layers
        weights = F.softmax(self.layer_weights, dim=0)
        
        smiles_emb = sum(w * h[:, 0, :] for w, h in zip(weights, hidden_states))
        
        # Descriptor embedding
        desc_emb = self.desc_proj(descriptors)
        
        # Gated fusion
        combined = torch.cat([smiles_emb, desc_emb], dim=-1)
        gate_values = self.gate(combined)
        
        # Apply gating
        gated_smiles = smiles_emb * gate_values
        gated_desc = desc_emb * (1 - gate_values)
        
        final_features = torch.cat([gated_smiles, gated_desc], dim=-1)
        
        return self.classifier(final_features).squeeze(-1)


# ============================================================================
# IMPROVEMENT 3: Focal Loss for Imbalanced Data
# ============================================================================

class FocalLoss(nn.Module):
    """
    Focal Loss: Focuses on hard examples
    Great for imbalanced bioactivity datasets
    """
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(
            inputs, targets.float(), reduction='none'
        )
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


# ============================================================================
# IMPROVEMENT 4: SMILES Augmentation
# ============================================================================

def augment_smiles(smiles_list, num_augmentations=3):
    """
    Augment SMILES strings by randomizing atom ordering
    This increases training data diversity
    
    Install: pip install rdkit
    """
    from rdkit import Chem
    
    augmented = []
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            # Original
            augmented.append(smiles)
            
            # Generate augmented versions
            for _ in range(num_augmentations):
                new_smiles = Chem.MolToSmiles(
                    mol, 
                    doRandom=True,
                    canonical=False
                )
                augmented.append(new_smiles)
        else:
            augmented.append(smiles)
    
    return augmented


class SMILESAugmentedDataset(torch.utils.data.Dataset):
    """
    Dataset with on-the-fly SMILES augmentation
    """
    def __init__(self, smiles_list, descriptors, labels, tokenizer, 
                 augment=True, num_aug=2):
        self.smiles_list = smiles_list
        self.descriptors = descriptors
        self.labels = labels
        self.tokenizer = tokenizer
        self.augment = augment
        self.num_aug = num_aug
        
    def __len__(self):
        return len(self.smiles_list)
    
    def __getitem__(self, idx):
        smiles = self.smiles_list[idx]
        
        # Augment during training
        if self.augment:
            from rdkit import Chem
            mol = Chem.MolFromSmiles(smiles)
            if mol and torch.rand(1).item() > 0.5:
                smiles = Chem.MolToSmiles(mol, doRandom=True, canonical=False)
        
        # Tokenize
        encoded = self.tokenizer(
            smiles,
            padding='max_length',
            truncation=True,
            max_length=512,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'descriptors': torch.tensor(self.descriptors[idx], dtype=torch.float32),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }


# ============================================================================
# IMPROVEMENT 5: Enhanced Training Strategy
# ============================================================================

def train_with_improvements(model, train_loader, val_loader, num_epochs=50):
    """
    Advanced training strategy for higher accuracy
    """
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import OneCycleLR
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    # Use focal loss for imbalanced data
    criterion = FocalLoss(alpha=0.25, gamma=2.0)
    
    # Separate learning rates for encoder and classifier
    encoder_params = []
    classifier_params = []
    
    for name, param in model.named_parameters():
        if param.requires_grad:
            if 'smiles_encoder' in name:
                encoder_params.append(param)
            else:
                classifier_params.append(param)
    
    optimizer = AdamW([
        {'params': encoder_params, 'lr': 1e-5},  # Lower LR for pretrained
        {'params': classifier_params, 'lr': 3e-4}  # Higher LR for new layers
    ], weight_decay=0.01)
    
    # OneCycleLR for better convergence
    scheduler = OneCycleLR(
        optimizer,
        max_lr=[1e-5, 3e-4],
        epochs=num_epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,  # 10% warmup
        anneal_strategy='cos'
    )
    
    best_val_acc = 0.0
    patience = 10
    patience_counter = 0
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            descriptors = batch['descriptors'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                descriptors=descriptors
            )
            
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).long()
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
        
        train_acc = 100 * train_correct / train_total
        
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0.0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                descriptors = batch['descriptors'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    descriptors=descriptors
                )
                
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = (torch.sigmoid(outputs) > 0.5).long()
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
        
        val_acc = 100 * val_correct / val_total
        
        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {train_loss/len(train_loader):.4f}, Acc: {train_acc:.2f}%")
        print(f"  Val Loss: {val_loss/len(val_loader):.4f}, Acc: {val_acc:.2f}%")
        
        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pt')
            print(f"  ✓ New best validation accuracy: {val_acc:.2f}%")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs")
                break
    
    return model, best_val_acc


# ============================================================================
# IMPROVEMENT 6: Test-Time Augmentation (TTA)
# ============================================================================

def predict_with_tta(model, smiles, descriptors, tokenizer, num_aug=5):
    """
    Test-time augmentation: Average predictions over augmented SMILES
    Expected gain: +0.5-1% accuracy
    """
    from rdkit import Chem
    
    device = next(model.parameters()).device
    model.eval()
    
    predictions = []
    
    # Original prediction
    encoded = tokenizer(smiles, padding='max_length', truncation=True, 
                       max_length=512, return_tensors='pt')
    
    with torch.no_grad():
        output = model(
            input_ids=encoded['input_ids'].to(device),
            attention_mask=encoded['attention_mask'].to(device),
            descriptors=torch.tensor(descriptors).unsqueeze(0).to(device)
        )
        predictions.append(torch.sigmoid(output).item())
    
    # Augmented predictions
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        for _ in range(num_aug):
            aug_smiles = Chem.MolToSmiles(mol, doRandom=True, canonical=False)
            encoded = tokenizer(aug_smiles, padding='max_length', 
                              truncation=True, max_length=512, return_tensors='pt')
            
            with torch.no_grad():
                output = model(
                    input_ids=encoded['input_ids'].to(device),
                    attention_mask=encoded['attention_mask'].to(device),
                    descriptors=torch.tensor(descriptors).unsqueeze(0).to(device)
                )
                predictions.append(torch.sigmoid(output).item())
    
    # Average predictions
    return sum(predictions) / len(predictions)


# ============================================================================
# IMPROVEMENT 7: Ensemble of Multiple Models
# ============================================================================

class EnsembleModel:
    """
    Ensemble multiple models for final prediction
    Expected gain: +1-2% accuracy
    """
    def __init__(self, models):
        self.models = models
        for model in self.models:
            model.eval()
    
    def predict(self, input_ids, attention_mask, descriptors):
        predictions = []
        
        with torch.no_grad():
            for model in self.models:
                output = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    descriptors=descriptors
                )
                predictions.append(torch.sigmoid(output))
        
        # Average predictions
        ensemble_pred = torch.stack(predictions).mean(dim=0)
        return ensemble_pred


# ============================================================================
# COMPLETE TRAINING PIPELINE
# ============================================================================

def complete_pipeline_example():
    """
    Complete pipeline from data to 95%+ accuracy
    """
    
    print("""
    COMPLETE PIPELINE TO ACHIEVE 95-97% ACCURACY
    =============================================
    
    Step 1: Data Preprocessing
    - Use more descriptors (8+ instead of 4)
    - Normalize/standardize descriptors
    - Handle class imbalance with SMOTE or class weights
    
    Step 2: Model Architecture
    - Use ImprovedMultimodalModel or MultiScaleMultimodalModel
    - Unfreeze 3-4 layers instead of 1-2
    - Use attention-based fusion
    
    Step 3: Training Strategy
    - Focal Loss for imbalanced data
    - Separate learning rates (1e-5 for encoder, 3e-4 for classifier)
    - OneCycleLR scheduler
    - SMILES augmentation during training
    - Gradient clipping
    - Early stopping
    
    Step 4: Inference
    - Test-time augmentation (TTA)
    - Ensemble 3-5 models trained with different seeds
    
    Expected Results:
    - Base model: 92%
    - + Better architecture: 93-94%
    - + Advanced training: 94-95%
    - + TTA: 95-96%
    - + Ensemble: 96-97%
    
    Quick Start:
    model = ImprovedMultimodalModel(
        num_desc=8,
        num_layers_to_unfreeze=3,
        dropout=0.3
    )
    
    # Train with improvements
    model, best_acc = train_with_improvements(
        model, train_loader, val_loader, num_epochs=50
    )
    """)


if __name__ == "__main__":
    complete_pipeline_example()


    COMPLETE PIPELINE TO ACHIEVE 95-97% ACCURACY

    Step 1: Data Preprocessing
    - Use more descriptors (8+ instead of 4)
    - Normalize/standardize descriptors
    - Handle class imbalance with SMOTE or class weights

    Step 2: Model Architecture
    - Use ImprovedMultimodalModel or MultiScaleMultimodalModel
    - Unfreeze 3-4 layers instead of 1-2
    - Use attention-based fusion

    Step 3: Training Strategy
    - Focal Loss for imbalanced data
    - Separate learning rates (1e-5 for encoder, 3e-4 for classifier)
    - OneCycleLR scheduler
    - SMILES augmentation during training
    - Gradient clipping
    - Early stopping

    Step 4: Inference
    - Test-time augmentation (TTA)
    - Ensemble 3-5 models trained with different seeds

    Expected Results:
    - Base model: 92%
    - + Better architecture: 93-94%
    - + Advanced training: 94-95%
    - + TTA: 95-96%
    - + Ensemble: 96-97%

    Quick Start:
    model = ImprovedMultimodalModel(
        num_desc=8,
       

In [49]:
num_epochs = 50

In [50]:
# Use focal loss for imbalanced data
from torch.optim.lr_scheduler import OneCycleLR
criterion = FocalLoss(alpha=0.25, gamma=2.0)

# Separate learning rates for encoder and classifier
encoder_params = []
classifier_params = []

for name, param in model.named_parameters():
    if param.requires_grad:
        if 'smiles_encoder' in name:
            encoder_params.append(param)
        else:
            classifier_params.append(param)

optimizer = AdamW([
    {'params': encoder_params, 'lr': 1e-5},  # Lower LR for pretrained
    {'params': classifier_params, 'lr': 3e-4}  # Higher LR for new layers
], weight_decay=0.01)

# OneCycleLR for better convergence
scheduler = OneCycleLR(
    optimizer,
    max_lr=[1e-5, 3e-4],
    epochs=num_epochs,
    steps_per_epoch=len(train_loader),
    pct_start=0.1,  # 10% warmup
    anneal_strategy='cos'
)

best_val_acc = 0.0
patience = 10
patience_counter = 0

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        descriptors = batch['descriptors'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            descriptors=descriptors
        )
        
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        train_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).long()
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
    
    train_acc = 100 * train_correct / train_total
    
    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0.0
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            descriptors = batch['descriptors'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                descriptors=descriptors
            )
            
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            preds = (torch.sigmoid(outputs) > 0.5).long()
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    
    val_acc = 100 * val_correct / val_total
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss/len(train_loader):.4f}, Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_loss/len(val_loader):.4f}, Acc: {val_acc:.2f}%")
    
    # Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"  ✓ New best validation accuracy: {val_acc:.2f}%")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break



Epoch 1/50
  Train Loss: 0.0309, Acc: 79.75%
  Val Loss: 0.0314, Acc: 81.83%
  ✓ New best validation accuracy: 81.83%
Epoch 2/50
  Train Loss: 0.0269, Acc: 83.90%
  Val Loss: 0.0320, Acc: 82.54%
  ✓ New best validation accuracy: 82.54%
Epoch 3/50
  Train Loss: 0.0252, Acc: 84.69%
  Val Loss: 0.0263, Acc: 83.77%
  ✓ New best validation accuracy: 83.77%
Epoch 4/50
  Train Loss: 0.0222, Acc: 86.57%
  Val Loss: 0.0280, Acc: 86.24%
  ✓ New best validation accuracy: 86.24%
Epoch 5/50
  Train Loss: 0.0199, Acc: 87.71%
  Val Loss: 0.0256, Acc: 86.95%
  ✓ New best validation accuracy: 86.95%
Epoch 6/50
  Train Loss: 0.0177, Acc: 89.65%
  Val Loss: 0.0260, Acc: 83.77%
Epoch 7/50
  Train Loss: 0.0153, Acc: 90.84%
  Val Loss: 0.0249, Acc: 88.36%
  ✓ New best validation accuracy: 88.36%
Epoch 8/50
  Train Loss: 0.0145, Acc: 91.90%
  Val Loss: 0.0259, Acc: 85.54%
Epoch 9/50
  Train Loss: 0.0134, Acc: 92.74%
  Val Loss: 0.0229, Acc: 89.42%
  ✓ New best validation accuracy: 89.42%
Epoch 10/50
  Train 

In [51]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Run inference on test set
model.eval()
test_preds = []
test_labels = []
test_probs = []
with torch.no_grad():
	for batch in test_loader:
		input_ids = batch['input_ids'].to(device)
		attention_mask = batch['attention_mask'].to(device)
		descriptors = batch['descriptors'].to(device)
		labels = batch['labels'].to(device)
		logits = model(input_ids, attention_mask, descriptors)
		probs = torch.sigmoid(logits).cpu().numpy()
		test_probs.extend(probs)
		test_preds.extend((probs > 0.5).astype(int))
		test_labels.extend(labels.cpu().numpy())

accuracy = balanced_accuracy_score(test_labels, test_preds)
auc = roc_auc_score(test_labels, test_probs)
print(f"AUC: {auc}")
print(f"Accuracy: {accuracy}")
print("accuracy another: ", accuracy_score(test_labels, test_preds))

AUC: 0.9208352402745996
Accuracy: 0.8334324942791762
accuracy another:  0.9118165784832452


In [26]:
complete_pipeline_example()


    COMPLETE PIPELINE TO ACHIEVE 95-97% ACCURACY

    Step 1: Data Preprocessing
    - Use more descriptors (8+ instead of 4)
    - Normalize/standardize descriptors
    - Handle class imbalance with SMOTE or class weights

    Step 2: Model Architecture
    - Use ImprovedMultimodalModel or MultiScaleMultimodalModel
    - Unfreeze 3-4 layers instead of 1-2
    - Use attention-based fusion

    Step 3: Training Strategy
    - Focal Loss for imbalanced data
    - Separate learning rates (1e-5 for encoder, 3e-4 for classifier)
    - OneCycleLR scheduler
    - SMILES augmentation during training
    - Gradient clipping
    - Early stopping

    Step 4: Inference
    - Test-time augmentation (TTA)
    - Ensemble 3-5 models trained with different seeds

    Expected Results:
    - Base model: 92%
    - + Better architecture: 93-94%
    - + Advanced training: 94-95%
    - + TTA: 95-96%
    - + Ensemble: 96-97%

    Quick Start:
    model = ImprovedMultimodalModel(
        num_desc=8,
       